# Voltage Control Benchmark: UI and API Usage
This benchmark lets you:
1. Download **training scenarios** (measurements + reference solutions).
2. Download **test scenarios** with anonymised timestamp keys.
3. Submit your **voltage control actions** (corrected voltages + adjusted loads).
4. Get a **score** per node / timestamp and an overall score.
Endpoints (with prefix):
- `/voltage_control_benchmark/training-scenarios`
- `/voltage_control_benchmark/voltage-control-data`
- `/voltage_control_benchmark/submit-results`
- `/voltage_control_benchmark/score`
- `/voltage_control_benchmark/ui` (web portal)

## 1. Using the Web UI
### 1.1. Open the Voltage Control Portal
1. In your browser, go to the home page:
   ```text
   http://localhost:8000/ (X)

2. Click the “Voltage Control” card:

You’ll see three main cards:
- 📥 Download Data
- 📤 Submit Results
- 📊 Check Score

### 1.2. Download data

In the **Download Data** card you have:
- `Voltage Limit` (numeric input, for example `230`)
- `Scenario` (`Overvoltages` / `Undervoltages`)
- `Difficulty` (`clean` / `easy` / `medium` / `hard`)
- **Download Training Scenarios**
- **Download Control Data**

![Downl. Data](images/vc_download_data.png)

#### Training scenarios (`/training-scenarios`)
When you click **Download Training Scenarios**:
- The UI opens:
  ```text
  /voltage_control_benchmark/training-scenarios?voltage_limit=<LIMIT>&Scenario=<Overvoltages|Undervoltages>

Important:
- `Voltage Limit` is used to select snapshots with violations
- `Scenario` chooses whether you are working with overvoltage or undervoltage cases
- `Difficulty` filters snapshots by **scenario hardness**
  - `clean`: no hardness filtering
  - `easy`, `medium`, `hard`: select subsets based on hardness quantiles

Internally, the training and control-data routers:

- query grids enabled for the voltage-control benchmark
- retrieve measurements matching the selected:
  - `voltage_limit`
  - `Scenario`
  - `difficulty`
- group measurements into snapshots by `(grid_id, datetime)`
- compute snapshot metrics and a `hardness_score`
- filter snapshots according to the requested difficulty

Each measurement tuple is:

(grid_id, node_id, phase, dt, power_active, power_reactive, voltage_magnitude, voltage_angle)

They are grouped as snapshots per (grid_id, datetime).

For each snapshot, it also fetches any previously stored solutions from "VoltageControlSolutions":

Return JSON structure for **training scenarios**:

```json
{
  "difficulty": "clean",
  "difficulty_profile": {
    "method": "none",
    "t1": 0.0,
    "t2": 0.0,
    "preference": "solutions_only",
    "preferred_population": 12
  },
  "scenario": "Overvoltages",
  "voltage_limit": 230.0,
  "grids": {
    "grid_1": {
      "2025-08-01T12:00:00": {
        "measurements": [
          {
            "node_id": "...",
            "phase": "R",
            "datetime": "...",
            "power_active": ...,
            "power_reactive": ...,
            "voltage_magnitude": ...,
            "voltage_angle": ...
          }
        ],
        "solutions": [
          {
            "node_id": "...",
            "phase": "R",
            "corrected_voltage": ...,
            "adjusted_power_active": ...,
            "adjusted_power_reactive": ...,
            "created_at": "..."
          }
        ],
        "metrics": {
          "hardness_score": ...
        }
      }
    }
  }
}

Use this for training and validation: you get both the scenario measurements and the stored reference solutions, together with per-snapshot metrics and hardness information.

#### Control data (`/voltage-control-data`)

When you click **Download Control Data**, the router:

- ensures the anonymised timestamp mapping table exists
- retrieves overvoltage or undervoltage measurements for the selected scenario
- groups them by `(grid_id, datetime)`
- computes hardness and filters snapshots according to `difficulty`
- builds persistent anonymised keys for the selected snapshots
- returns measurements grouped as:

`grid_id -> anonymised_key -> list[measurement]`

Reference solutions are intentionally hidden in this endpoint.

Return JSON:

```json
{
  "difficulty": "medium",
  "difficulty_profile": {
    "method": "quantiles_33_66",
    "t1": 0.12,
    "t2": 0.41,
    "preference": "solutions_only",
    "preferred_population": 20
  },
  "scenario": "Overvoltages",
  "voltage_limit": 230.0,
  "grids": {
    "grid_1": {
      "ANON_KEY_1": [
        {
          "node_id": "...",
          "phase": "R",
          "power_active": ...,
          "power_reactive": ...,
          "voltage_magnitude": ...,
          "voltage_angle": ...
        }
      ]
    }
  }
}

Each anon_key corresponds to one snapshot in time for that grid.

Your job is to propose voltage corrections and load adjustments per node for each anonymised snapshot.

### 1.3. Submit results (UI)
In the **“📤 Submit Results”** card:
- `User ID`
- `Grid ID`
- File input (JSON)
- **Submit Results** button

![Submit results](images/vc_submit_results.png)

Expected **JSON file content** (this is the `guesses` part of `VCGuessSubmission`):

```json
{
  "ANON_KEY_1": [
    {
      "node_id": "N001",
      "phase": "R",
      "corrected_voltage": 230.0,
      "adjusted_power_active": 4.5,
      "adjusted_power_reactive": 1.2
    },
    {
      "node_id": "N002",
      "phase": "R",
      "corrected_voltage": 228.0,
      "adjusted_power_active": 3.8,
      "adjusted_power_reactive": 1.0
    }
  ],
  "ANON_KEY_2": [
    ...
  ]
}

The UI wraps this as:

```json
{
  "guess_id": "<User ID>",
  "grid_id": "<Grid ID>",
  "guesses": {
    "ANON_KEY_1": [ ... ],
    "ANON_KEY_2": [ ... ]
  }
}

and POSTs to:

/voltage_control_benchmark/submit-results

The router:
- Ensures guesses table exists.
- For each (anon_key, node_solutions):
    - Builds guessed_voltages
    - Builds guessed_loads
    - Checks if a row already exists (to track duplicates).
    - Upserts into guesses table
    
On success, response:

```json
{
  "status": "submitted",
  "summary": {
    "submitted": <n_new>,
    "duplicates_skipped": <n_overwritten>
  }
}

### 1.4. Check score (UI)

In the **Check Score** card:
- `User ID`
- `Grid ID`
- **Check Score**

![Check scores](images/vc_check_score.png)

The UI calls:
```text
/voltage_control_benchmark/score?guess_id=<UserID>&grid_id=<GridID>

The router:

1. joins your guesses with the anonymised timestamp mapping table
2. for each `(anonymised_key, datetime)`:
   - retrieves the original measurements
   - retrieves reference voltage-control solutions if available
   - compares your corrected voltages with the reference voltages
   - computes an effort penalty based on total `|ΔP|`
3. builds a per-timestamp score
4. aggregates all timestamp scores into an `overall_score`

Depending on the available reference information, each timestamp result may use a different scoring mode, such as:
- `ref_voltage_then_effort`
- `ref_voltage_miss_penalised`
- `outcome_only_fallback`

Response:

```json
{
  "guess_id": "user_1",
  "grid_id": "grid_1",
  "total_nodes": 123,
  "overall_score": 0.82,
  "per_timestamp": [
    {
      "anonymised_key": "ANON_KEY_1",
      "datetime": "...",
      "nodes_total": 10,
      "mode": "ref_voltage_then_effort",
      "reference_available": true,
      "reference_voltage_mae": 0.42,
      "reference_voltage_tol": 1.0,
      "voltage_factor": 1.0,
      "effort_norm": 0.08,
      "effort_factor": 0.93,
      "timestamp_score": 0.93,
      "average_node_score_diagnostic": 0.77,
      "reference_voltage_coverage": 1.0,
      "nodes": [
        {
          "node_id": "...",
          "phase": "R",
          "measured_voltage": 236.1,
          "corrected_voltage": 230.2,
          "measured_power_active": 12.5,
          "corrected_power_active": 11.9,
          "voltage_score": 0.8,
          "power_penalty": 0.95,
          "bonus": 1.0,
          "node_score": 0.865,
          "reference_corrected_voltage": 230.0
        }
      ]
    }
  ]
}

The UI shows:
- overall score
- total nodes
- per-timestamp summaries including:
  - timestamp key
  - node count
  - effort metrics
  - timestamp score

The raw API response contains more detail than the UI summary, including per-node diagnostics.

## 2. Using the API from Python
Now let’s do the same from a notebook using `requests`.

In [ ]:
import requests
import json
BASE_URL = "http://localhost:8000"
def check_response(resp: requests.Response):
    """Raise on error and return parsed JSON or text."""
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        try:
            print("Error payload:", resp.json())
        except Exception:
            print("Raw response:", resp.text)
        raise e
    try:
        return resp.json()
    except Exception:
        return resp.text

### 2.1. Get training scenarios (`GET /training-scenarios`)

In [ ]:
def get_training_voltage_control_data(
    voltage_limit: float,
    scenario: str = "Overvoltages",
    difficulty: str = "clean",
):
    """
    scenario: "Overvoltages" or "Undervoltages"
    difficulty: "clean", "easy", "medium", or "hard"
    """
    url = f"{BASE_URL}/voltage_control_benchmark/training-scenarios"
    params = {
        "voltage_limit": voltage_limit,
        "Scenario": scenario,
        "difficulty": difficulty,
    }
    return check_response(requests.get(url, params=params))

# Example:
# training_data = get_training_voltage_control_data(
#     voltage_limit=230.0,
#     scenario="Overvoltages",
#     difficulty="clean",
# )

### 2.2. Get control data (`GET /voltage-control-data`)

In [ ]:
def get_voltage_control_data(
    voltage_limit: float,
    scenario: str = "Overvoltages",
    difficulty: str = "clean",
):
    url = f"{BASE_URL}/voltage_control_benchmark/voltage-control-data"
    params = {
        "voltage_limit": voltage_limit,
        "Scenario": scenario,
        "difficulty": difficulty,
    }
    return check_response(requests.get(url, params=params))

# Example:
# control_data = get_voltage_control_data(
#     voltage_limit=230.0,
#     scenario="Overvoltages",
#     difficulty="medium",
# )

### 2.3. Submit guesses (`POST /submit-results`)
Pydantic `VCGuessSubmission` roughly:
```python
class VCGuessSubmission(BaseModel):
    guess_id: str
    grid_id: str
    guesses: Dict[str, List[VoltageControlSolution]]
class VoltageControlSolution(BaseModel):
    node_id: str
    phase: str
    corrected_voltage: float
    adjusted_power_active: float
    adjusted_power_reactive: float

So your JSON must look like:

```json
{
  "guess_id": "user_1",
  "grid_id": "grid_1",
  "guesses": {
    "ANON_KEY_1": [ {...}, {...} ],
    "ANON_KEY_2": [ ... ]
  }
}

We’ll wrap it:

In [ ]:
def submit_voltage_control_guesses(
    guess_id: str,
    grid_id: str,
    guesses: dict[str, list[dict]],
):
    """
    guesses: dict[anon_key] -> list of dicts with
    {
      "node_id": str,
      "phase": "R"/"S"/"T",
      "corrected_voltage": float,
      "adjusted_power_active": float,
      "adjusted_power_reactive": float
    }
    """
    url = f"{BASE_URL}/voltage_control_benchmark/submit-results"
    payload = {"guess_id": guess_id, "grid_id": grid_id, "guesses": guesses}
    resp = requests.post(url, json=payload)
    return check_response(resp)
# Example dummy payload:
# guesses = {
#   "ANON_KEY_1": [
#     {
#       "node_id": "N001",
#       "phase": "R",
#       "corrected_voltage": 230.0,
#       "adjusted_power_active": 4.5,
#       "adjusted_power_reactive": 1.0
#     }
#   ]
# }
# submit_voltage_control_guesses("user_1", "grid_1", guesses)

The router aggregates and stores your guesses in `VCUserGuesses`, and returns how many anonymised snapshots were newly inserted versus overwritten.

### 2.4. Get score (`GET /score`)

In [ ]:
def get_voltage_control_score(guess_id: str, grid_id: str):
    url = f"{BASE_URL}/voltage_control_benchmark/score"
    params = {"guess_id": guess_id, "grid_id": grid_id}
    return check_response(requests.get(url, params=params))
# Example:
# score = get_voltage_control_score("user_1", "grid_1")
# score

You’ll get:

```json
{
  "guess_id": "user_1",
  "grid_id": "grid_1",
  "total_nodes": 123,
  "overall_score": 0.82,
  "per_timestamp": [
    {
      "anonymised_key": "ANON_KEY_1",
      "datetime": "...",
      "nodes_total": 10,
      "mode": "ref_voltage_then_effort",
      "reference_available": true,
      "reference_voltage_mae": 0.42,
      "reference_voltage_tol": 1.0,
      "voltage_factor": 1.0,
      "effort_norm": 0.08,
      "effort_factor": 0.93,
      "timestamp_score": 0.93,
      "average_node_score_diagnostic": 0.77,
      "reference_voltage_coverage": 1.0,
      "nodes": [ ... ]
    }
  ]
}

You can then analyse where your algorithm performs well or poorly:
- snapshot by snapshot
- node by node
- reference-voltage tracking versus control effort

## 3. Example end-to-end pipeline

Below is a small example tying everything together.

### 3.1. Get control data and pick a snapshot

In [ ]:
control_data = get_voltage_control_data(
    voltage_limit=230.0,
    scenario="Overvoltages",
    difficulty="clean",
)

grid_ids = list(control_data["grids"].keys())
grid_ids

Pick a grid and an anonymised snapshot:

In [ ]:
grid_id = grid_ids[0]
snapshots = control_data["grids"][grid_id]
anon_keys = list(snapshots.keys())
anon_keys[:5]

In [ ]:
anon_key = anon_keys[0]
snapshot_measurements = snapshots[anon_key]
snapshot_measurements[:3]

### 3.2. Dummy voltage control algorithm
**Replace this with your real controller.**
Here we simply:
- Set `corrected_voltage` to nominal `230.0` for every node.
- Leave active/reactive powers unchanged.

In [ ]:
def naive_voltage_controller(measurements: list[dict]) -> list[dict]:
    nominal = 230.0
    solutions: list[dict] = []
    for m in measurements:
        solutions.append(
            {
                "node_id": m["node_id"],
                "phase": m["phase"],
                "corrected_voltage": nominal,
                "adjusted_power_active": m["power_active"],
                "adjusted_power_reactive": m["power_reactive"],
            }
        )
    return solutions
solutions_for_snapshot = naive_voltage_controller(snapshot_measurements)
solutions_for_snapshot[:3]

Build guesses for all snapshots of that grid using this naive strategy:

In [ ]:
guesses = {}
for ak, meas in snapshots.items():
    guesses[ak] = naive_voltage_controller(meas)
len(guesses), list(guesses.keys())[:3]

### 3.3. Submit and score

In [ ]:
guess_id = "vc_demo_user_01"
submit_res = submit_voltage_control_guesses(
    guess_id=guess_id,
    grid_id=grid_id,
    guesses=guesses,
)
submit_res

In [ ]:
score = get_voltage_control_score(guess_id, grid_id)

score

You now have an `overall_score` plus a detailed per-timestamp and per-node breakdown.

Improve your controller and resubmit to see how the score changes.

## 4. Summary

- **Training scenarios** (`/training-scenarios`)
  - return measurements, reference solutions, and per-snapshot metrics
  - require `voltage_limit`
  - support `Scenario` and `difficulty`

- **Control data** (`/voltage-control-data`)
  - return evaluation snapshots keyed by anonymised timestamp keys
  - require `voltage_limit`
  - support `Scenario` and `difficulty`

- **Submit results** (`/submit-results`)
  - submit guessed corrected voltages and power adjustments per anonymised snapshot

- **Score** (`/score`)
  - returns `overall_score`
  - includes detailed `per_timestamp` diagnostics
  - combines reference-voltage quality and control effort


You can explore everything via:
- the **Voltage Control Portal** at `/voltage_control_benchmark/ui`, or
- the **API** using the Python snippets above.

---